# HW2 Part 2 - Sherlock Holmes SFT with LoRA

This notebook contains the cluster-side workflow for HW2. It assumes the local data-generation phase has already been completed and that the prepared `data/processed/*.jsonl` files were copied to the cluster.

The training pipeline is: base Qwen instruct model -> merge HW1 Sherlock LoRA adapter -> train fresh SFT LoRA adapters -> evaluate zero-shot and three-shot configurations -> run ablations.

## 0. Cluster Session

RTX 3090 command used for the run:

```bash
sjupyter --time 0-8:00:00 --qos course --part rtx3090 --gpu rtx_3090:1 --account cours_ibm26
```

All notebook cells should be run from the `~/HW2` project folder on the GPU node.

In [ ]:
!hostname
!nvidia-smi

## 1. Configuration and Run Controls

The project configuration remains in `hw2_config.json`. The local generation stage is intentionally not repeated here; this notebook uses the prepared dataset files.

In [ ]:
from pathlib import Path
import collections
import gc
import inspect
import json
import math
import os
import random
import re
import string
from typing import Any, Dict, List, Optional, Sequence, Tuple

# Text-only project: avoid optional vision/audio backends that can break when torchvision is mismatched.
os.environ.setdefault('TRANSFORMERS_NO_TORCHVISION', '1')
os.environ.setdefault('TRANSFORMERS_NO_VISION', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / 'hw2_config.json'
CONFIG = json.loads(CONFIG_PATH.read_text())
SEED = int(CONFIG['project']['seed'])
random.seed(SEED)

RUN_DATASET_CHECKS = True
RUN_MERGE_PART1 = True
TRAIN_EXPERIMENTS = ['main']          # Add ablations here after main succeeds.
EVALUATE_REQUIRED = True
EVALUATE_ABLATIONS = []               # Example: ['rank8_sft', 'full_sequence_loss']
RUN_OOD_PPL = False                   # Set True after the OOD text is ready.
MAKE_REPORT_ASSETS = True
FORCE_RETRAIN = False                 # Keep False to avoid overwriting completed adapters.

# TRL assistant_only_loss requires a chat template with assistant-token masks.
# Qwen's template in this cluster environment does not expose that mask, so the
# executable notebook uses full-sequence SFT and records this as a limitation.
CONFIG['sft_defaults']['assistant_only_loss'] = False

print(json.dumps(CONFIG['project'], indent=2))
print('Project root:', PROJECT_ROOT)

## 2. Environment Check

Use the current Jupyter kernel environment. If `torch` is missing, run `%pip install -r requirements_cluster.txt` in this notebook kernel before continuing.

In [ ]:
import sys
print('Python executable:', sys.executable)
print('Python version:', sys.version)

# Uncomment if the current kernel is missing dependencies.
# %pip install -r requirements_cluster.txt
print('TRANSFORMERS_NO_TORCHVISION:', os.environ.get('TRANSFORMERS_NO_TORCHVISION'))
print('TRANSFORMERS_NO_VISION:', os.environ.get('TRANSFORMERS_NO_VISION'))

# If PEFT/Transformers later fails with torchvision::nms, run in a notebook cell:
# %pip uninstall -y torchvision
# then restart the kernel and rerun from the top.


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA is not available. Use the RTX 3090 Jupyter session.'

## 3. General Utilities

Small dependency-light helpers for JSONL, metadata parsing, answer normalization, and report tables.

In [ ]:
def project_path(path: str | Path) -> Path:
    p = Path(path)
    return p if p.is_absolute() else PROJECT_ROOT / p


def read_jsonl(path: str | Path) -> List[Dict[str, Any]]:
    p = project_path(path)
    rows = []
    if not p.exists():
        return rows
    with p.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path: str | Path, rows: Sequence[Dict[str, Any]]) -> None:
    p = project_path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def parse_notes(notes: Optional[str]) -> Dict[str, str]:
    meta = {}
    if not notes:
        return meta
    for part in str(notes).split(';'):
        part = part.strip()
        if '=' in part:
            key, value = part.split('=', 1)
            meta[key.strip()] = value.strip()
    return meta


def message_by_role(row: Dict[str, Any], role: str) -> str:
    for msg in row.get('messages', []):
        if msg.get('role') == role:
            return str(msg.get('content', ''))
    return ''


def validate_messages(row: Dict[str, Any]) -> Tuple[bool, str]:
    messages = row.get('messages')
    if not isinstance(messages, list) or len(messages) < 3:
        return False, 'messages must contain system, user, assistant'
    roles = [m.get('role') for m in messages[:3]]
    if roles != ['system', 'user', 'assistant']:
        return False, f'unexpected roles: {roles}'
    for msg in messages[:3]:
        if not isinstance(msg.get('content'), str) or not msg['content'].strip():
            return False, f'empty {msg.get("role")} content'
    return True, ''


def extract_book(row: Dict[str, Any]) -> str:
    system = message_by_role(row, 'system')
    match = re.search(r'based on:\s*(.*?)(?:\.?$)', system, flags=re.I)
    if match:
        return match.group(1).strip().rstrip('.')
    if re.search(r'based on the text', system, flags=re.I):
        return 'Not specified (provided unanswerable)'
    return parse_notes(row.get('notes')).get('book', 'Not specified')


def extract_category(row: Dict[str, Any]) -> str:
    return parse_notes(row.get('notes')).get('category', 'unknown')


def extract_source(row: Dict[str, Any]) -> str:
    return parse_notes(row.get('notes')).get('source', 'unknown')


def normalize_answer(text: str) -> str:
    text = text.lower()
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    return ' '.join(text.split())


def token_f1(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    if not pred_tokens and not truth_tokens:
        return 1.0
    if not pred_tokens or not truth_tokens:
        return 0.0
    common = collections.Counter(pred_tokens) & collections.Counter(truth_tokens)
    same = sum(common.values())
    if same == 0:
        return 0.0
    precision = same / len(pred_tokens)
    recall = same / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)


def extract_aliases(row: Dict[str, Any]) -> List[str]:
    aliases = []
    meta = parse_notes(row.get('notes'))
    for key in ['alt_answers', 'aliases', 'alt_answers_nqa']:
        value = row.get(key)
        if isinstance(value, list):
            aliases.extend(str(v) for v in value)
        elif isinstance(value, str):
            aliases.append(value)
        raw = meta.get(key)
        if raw:
            if raw.startswith('['):
                try:
                    aliases.extend(str(v) for v in json.loads(raw))
                    continue
                except json.JSONDecodeError:
                    pass
            aliases.extend(x.strip().strip("\"'") for x in raw.split('|') if x.strip())
    answer = message_by_role(row, 'assistant')
    if answer:
        aliases.append(answer)
    unique, seen = [], set()
    for alias in aliases:
        key = normalize_answer(alias)
        if key and key not in seen:
            seen.add(key)
            unique.append(alias)
    return unique


def score_prediction(prediction: str, answers: Sequence[str]) -> Dict[str, float]:
    normalized_prediction = normalize_answer(prediction)
    normalized_answers = [normalize_answer(a) for a in answers if normalize_answer(a)]
    accuracy = 1.0 if normalized_prediction in normalized_answers else 0.0
    f1 = max((token_f1(prediction, answer) for answer in answers), default=0.0)
    return {'accuracy': accuracy, 'f1': f1}


def markdown_table(headers: Sequence[str], rows: Sequence[Sequence[Any]]) -> str:
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    for row in rows:
        lines.append('| ' + ' | '.join(str(x) for x in row) + ' |')
    return '\n'.join(lines)


## 4. Load Prepared Dataset

The prepared files are the output of the local run. This notebook does not regenerate local Ollama data.

In [ ]:
train_rows = read_jsonl(CONFIG['dataset']['train_messages'])
validation_rows = read_jsonl(CONFIG['dataset']['validation_messages'])
test_rows = read_jsonl(CONFIG['dataset']['test_messages'])
test_answer_rows = read_jsonl(CONFIG['dataset']['test_answers'])

assert train_rows, 'Missing train_messages.jsonl. Copy data/processed from the local run.'
assert validation_rows, 'Missing validation_messages.jsonl.'
assert test_rows, 'Missing test_messages.jsonl.'

if RUN_DATASET_CHECKS:
    bad_rows = []
    for split_name, rows in [('train', train_rows), ('validation', validation_rows), ('test', test_rows)]:
        for i, row in enumerate(rows):
            ok, reason = validate_messages(row)
            if not ok:
                bad_rows.append((split_name, i, reason))
    assert not bad_rows, bad_rows[:10]

print('train:', len(train_rows))
print('validation:', len(validation_rows))
print('test:', len(test_rows))
print('test answers:', len(test_answer_rows))

## 5. Dataset Diagnostics

These are report-ready distribution tables for the dataset card.

In [ ]:
import pandas as pd

all_rows = train_rows + validation_rows + test_rows
split_df = pd.DataFrame([
    {'split': 'train', 'count': len(train_rows)},
    {'split': 'validation', 'count': len(validation_rows)},
    {'split': 'test', 'count': len(test_rows)},
])
category_df = pd.DataFrame(collections.Counter(extract_category(r) for r in all_rows).most_common(), columns=['category', 'count'])
book_df = pd.DataFrame(collections.Counter(extract_book(r) for r in all_rows).most_common(), columns=['book', 'count'])
source_df = pd.DataFrame(collections.Counter(extract_source(r) for r in all_rows).most_common(), columns=['source', 'count'])

print(split_df)
print(category_df)
print(book_df)
print(source_df)

## 6. Chat Template Snapshot

The assignment asks to show how the OpenAI-style `messages` field becomes model text. This snapshot uses the tokenizer's chat template.

In [ ]:
from transformers import AutoTokenizer

tokenizer_preview = AutoTokenizer.from_pretrained(CONFIG['model']['base_model_id'], trust_remote_code=CONFIG['model'].get('trust_remote_code', False))
sample = train_rows[0]
print(json.dumps(sample['messages'], indent=2, ensure_ascii=False))
print('\n--- chat template ---')
print(tokenizer_preview.apply_chat_template(sample['messages'], tokenize=False, add_generation_prompt=False))

## 7. Model and Training Helpers

The Part 1 LoRA adapter is merged into the base model. Each Part 2 experiment then trains a fresh LoRA adapter while keeping the model weights frozen.

In [ ]:
from datasets import Dataset
import transformers

# Compatibility shim for PEFT/Transformers combinations on the cluster.
# Some Transformers versions expose BloomPreTrainedModel only from the BLOOM module,
# while PEFT imports it from the top-level transformers namespace.
if not hasattr(transformers, 'BloomPreTrainedModel'):
    from transformers.models.bloom.modeling_bloom import BloomPreTrainedModel
    transformers.BloomPreTrainedModel = BloomPreTrainedModel

from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


def torch_dtype_from_config(name: str):
    return {'float16': torch.float16, 'bfloat16': torch.bfloat16, 'float32': torch.float32}[name]


def bnb_config(config: Dict[str, Any]) -> BitsAndBytesConfig:
    q = config['quantization']
    return BitsAndBytesConfig(
        load_in_4bit=q['load_in_4bit'],
        bnb_4bit_quant_type=q['bnb_4bit_quant_type'],
        bnb_4bit_use_double_quant=q['bnb_4bit_use_double_quant'],
        bnb_4bit_compute_dtype=torch_dtype_from_config(q['bnb_4bit_compute_dtype']),
    )


def load_tokenizer(model_id_or_path: str, config: Dict[str, Any], padding_side: str = 'right'):
    tokenizer = AutoTokenizer.from_pretrained(model_id_or_path, trust_remote_code=config['model'].get('trust_remote_code', False))
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = padding_side
    return tokenizer


def merge_part1_adapter(config: Dict[str, Any]) -> Path:
    merged_dir = project_path(config['paths']['part1_merged_model_dir'])
    if (merged_dir / 'config.json').exists():
        print('Part 1 merged model already exists:', merged_dir)
        return merged_dir
    adapter_dir = Path(config['paths']['hw1_best_adapter'])
    assert adapter_dir.exists(), f'Missing HW1 adapter: {adapter_dir}'
    merged_dir.mkdir(parents=True, exist_ok=True)
    base = AutoModelForCausalLM.from_pretrained(
        config['model']['base_model_id'],
        torch_dtype=torch_dtype_from_config(config['model']['torch_dtype']),
        device_map=config['model']['device_map'],
        trust_remote_code=config['model'].get('trust_remote_code', False),
        attn_implementation=config['model'].get('attn_implementation', 'sdpa'),
    )
    tokenizer = load_tokenizer(config['model']['base_model_id'], config)
    peft_model = PeftModel.from_pretrained(base, adapter_dir)
    merged = peft_model.merge_and_unload(safe_merge=True)
    merged.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    del merged, peft_model, base
    gc.collect()
    torch.cuda.empty_cache()
    print('Saved merged Part 1 model:', merged_dir)
    return merged_dir


def merge_dict(base: Dict[str, Any], overrides: Dict[str, Any]) -> Dict[str, Any]:
    merged = dict(base)
    merged.update(overrides or {})
    return merged


def compatible_kwargs(callable_obj, kwargs: Dict[str, Any]) -> Dict[str, Any]:
    params = inspect.signature(callable_obj).parameters
    return {k: v for k, v in kwargs.items() if k in params}


def model_path_for_experiment(config: Dict[str, Any], exp_cfg: Dict[str, Any]) -> str:
    if exp_cfg['base_stage'] == 'base_model':
        return config['model']['base_model_id']
    return str(project_path(config['paths']['part1_merged_model_dir']))

## 8. Merge Part 1 Adapter

This creates the intended starting checkpoint for Part 2: the base model with the HW1 Sherlock adaptation merged in.

In [ ]:
if RUN_MERGE_PART1:
    part1_model_dir = merge_part1_adapter(CONFIG)
else:
    part1_model_dir = project_path(CONFIG['paths']['part1_merged_model_dir'])
part1_model_dir

## 9. SFT Training

The main run uses rank 16, alpha 32, attention+MLP targets, 512-token examples, learning rate `5e-5`, and full-sequence loss. Assistant-only loss was planned, but TRL requires a chat template that emits assistant-token masks (`{% generation %}`), which the available Qwen template does not provide in this environment. This is recorded as a training limitation; ablations override one design choice at a time.

In [ ]:
def build_sft_config(output_dir: Path, training_cfg: Dict[str, Any]) -> SFTConfig:
    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=training_cfg['learning_rate'],
        num_train_epochs=training_cfg['num_train_epochs'],
        per_device_train_batch_size=training_cfg['per_device_train_batch_size'],
        per_device_eval_batch_size=training_cfg['per_device_eval_batch_size'],
        gradient_accumulation_steps=training_cfg['gradient_accumulation_steps'],
        lr_scheduler_type=training_cfg['lr_scheduler_type'],
        warmup_ratio=training_cfg['warmup_ratio'],
        optim=training_cfg['optim'],
        weight_decay=training_cfg['weight_decay'],
        max_grad_norm=training_cfg['max_grad_norm'],
        logging_steps=training_cfg['logging_steps'],
        eval_strategy='steps',
        eval_steps=training_cfg['eval_steps'],
        save_steps=training_cfg['save_steps'],
        save_total_limit=training_cfg['save_total_limit'],
        packing=training_cfg['packing'],
        assistant_only_loss=training_cfg['assistant_only_loss'],
        gradient_checkpointing=training_cfg.get('gradient_checkpointing', True),
        report_to=training_cfg['report_to'],
        seed=SEED,
    )
    params = inspect.signature(SFTConfig).parameters
    if 'max_length' in params:
        kwargs['max_length'] = training_cfg['max_seq_length']
    else:
        kwargs['max_seq_length'] = training_cfg['max_seq_length']
    if 'eval_strategy' not in params and 'evaluation_strategy' in params:
        kwargs['evaluation_strategy'] = kwargs.pop('eval_strategy')
    return SFTConfig(**compatible_kwargs(SFTConfig, kwargs))


def train_sft_experiment(experiment: str, config: Dict[str, Any]) -> Path:
    exp_cfg = config['experiments'][experiment]
    final_adapter_dir = project_path(config['paths']['output_root']) / 'sft' / experiment / 'final_adapter'
    if final_adapter_dir.exists() and not FORCE_RETRAIN:
        print('Adapter already exists, skipping:', final_adapter_dir)
        return final_adapter_dir
    if exp_cfg['base_stage'] == 'part1_merged':
        merge_part1_adapter(config)
    train_path = f"data/processed/train_{exp_cfg['dataset_variant']}.jsonl"
    train_data = read_jsonl(train_path)
    eval_data = read_jsonl(config['dataset']['validation_messages'])
    assert train_data, f'No training rows found in {train_path}'
    output_dir = project_path(config['paths']['output_root']) / 'sft' / experiment
    output_dir.mkdir(parents=True, exist_ok=True)
    training_cfg = merge_dict(config['sft_defaults'], exp_cfg.get('training', {}))
    lora_cfg = merge_dict(config['lora_defaults'], exp_cfg.get('lora', {}))
    model_path = model_path_for_experiment(config, exp_cfg)
    print('Experiment:', experiment)
    print(json.dumps({'training': training_cfg, 'lora': lora_cfg}, indent=2))
    tokenizer = load_tokenizer(model_path, config, padding_side='right')
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config(config),
        device_map=config['model']['device_map'],
        trust_remote_code=config['model'].get('trust_remote_code', False),
        attn_implementation=config['model'].get('attn_implementation', 'sdpa'),
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(
        r=lora_cfg['r'],
        lora_alpha=lora_cfg['lora_alpha'],
        lora_dropout=lora_cfg['lora_dropout'],
        bias=lora_cfg['bias'],
        task_type=lora_cfg['task_type'],
        target_modules=lora_cfg['target_modules'],
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    args = build_sft_config(output_dir, training_cfg)
    trainer_kwargs = dict(model=model, args=args, train_dataset=Dataset.from_list(train_data), eval_dataset=Dataset.from_list(eval_data))
    if 'processing_class' in inspect.signature(SFTTrainer).parameters:
        trainer_kwargs['processing_class'] = tokenizer
    else:
        trainer_kwargs['tokenizer'] = tokenizer
    trainer = SFTTrainer(**trainer_kwargs)
    trainer.train()
    trainer.save_model(str(final_adapter_dir))
    tokenizer.save_pretrained(final_adapter_dir)
    (output_dir / 'experiment_config.json').write_text(json.dumps({'experiment': experiment, 'training': training_cfg, 'lora': lora_cfg, 'source_config': exp_cfg}, indent=2), encoding='utf-8')
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return final_adapter_dir

In [ ]:
trained_adapters = {}
for experiment in TRAIN_EXPERIMENTS:
    trained_adapters[experiment] = train_sft_experiment(experiment, CONFIG)
trained_adapters

## 10. Evaluation Functions

All required systems are evaluated on the same held-out test set. Metrics are normalized exact-match accuracy and token-level F1.

In [ ]:
def load_eval_model(config: Dict[str, Any], base_stage: str, adapter_experiment: Optional[str]):
    model_path = config['model']['base_model_id'] if base_stage == 'base_model' else str(project_path(config['paths']['part1_merged_model_dir']))
    tokenizer = load_tokenizer(model_path, config, padding_side='left')
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config(config),
        device_map=config['model']['device_map'],
        trust_remote_code=config['model'].get('trust_remote_code', False),
        attn_implementation=config['model'].get('attn_implementation', 'sdpa'),
    )
    if adapter_experiment:
        adapter_path = project_path(config['paths']['output_root']) / 'sft' / adapter_experiment / 'final_adapter'
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return model, tokenizer


def build_eval_messages(row: Dict[str, Any], shot_rows: List[Dict[str, Any]], config: Dict[str, Any]) -> List[Dict[str, str]]:
    system = message_by_role(row, 'system')
    instruction = config['evaluation']['prompt_instruction']
    messages = [{'role': 'system', 'content': f'{system}\n\n{instruction}'}]
    for shot in shot_rows:
        messages.append({'role': 'user', 'content': message_by_role(shot, 'user')})
        messages.append({'role': 'assistant', 'content': message_by_role(shot, 'assistant')})
    messages.append({'role': 'user', 'content': message_by_role(row, 'user')})
    return messages


def generate_answer(model, tokenizer, messages: List[Dict[str, str]], config: Dict[str, Any]) -> str:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    gen_cfg = config['evaluation']
    kwargs = dict(max_new_tokens=gen_cfg['max_new_tokens'], do_sample=gen_cfg['do_sample'], pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    if gen_cfg['do_sample']:
        kwargs['temperature'] = gen_cfg['temperature']
    with torch.no_grad():
        output_ids = model.generate(**inputs, **kwargs)
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate_config(name: str, eval_cfg: Dict[str, Any], config: Dict[str, Any]) -> Dict[str, Any]:
    shot_rows = train_rows[:int(eval_cfg['shots'])]
    model, tokenizer = load_eval_model(config, eval_cfg['base_stage'], eval_cfg.get('adapter_experiment'))
    predictions = []
    for i, row in enumerate(test_rows):
        pred = generate_answer(model, tokenizer, build_eval_messages(row, shot_rows, config), config)
        answers = extract_aliases(row)
        score = score_prediction(pred, answers) if pred.strip() else {'accuracy': 0.0, 'f1': 0.0}
        predictions.append({'id': f'test_{i:04d}', 'eval_config': name, 'question': message_by_role(row, 'user'), 'gold_answers': answers, 'prediction': pred, 'accuracy': score['accuracy'], 'f1': score['f1']})
        if (i + 1) % 10 == 0 or i == len(test_rows) - 1:
            print(f'[{name}] {i + 1}/{len(test_rows)}')
    summary = {'eval_config': name, 'accuracy': sum(p['accuracy'] for p in predictions) / len(predictions), 'f1': sum(p['f1'] for p in predictions) / len(predictions), 'n': len(predictions)}
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {'summary': summary, 'predictions': predictions}


def save_eval_outputs(tag: str, results: List[Dict[str, Any]], config: Dict[str, Any]) -> pd.DataFrame:
    out_dir = project_path(config['paths']['output_root']) / 'evaluation'
    out_dir.mkdir(parents=True, exist_ok=True)
    summaries = [r['summary'] for r in results]
    predictions = [p for r in results for p in r['predictions']]
    summary_name = 'summary.json' if tag == 'required' else f'summary_{tag}.json'
    pred_name = 'predictions_required.jsonl' if tag == 'required' else f'predictions_{tag}.jsonl'
    (out_dir / summary_name).write_text(json.dumps(summaries, indent=2), encoding='utf-8')
    write_jsonl(out_dir / pred_name, predictions)
    return pd.DataFrame(summaries)

## 11. Required Four-Way Evaluation

This is the assignment's main comparison: Part 1 zero-shot, Part 1 three-shot, Part 2 zero-shot, and Part 2 three-shot.

In [ ]:
if EVALUATE_REQUIRED:
    required_results = []
    for name, eval_cfg in CONFIG['evaluation']['required_configs'].items():
        required_results.append(evaluate_config(name, eval_cfg, CONFIG))
    required_eval_df = save_eval_outputs('required', required_results, CONFIG)
else:
    summary_path = project_path(CONFIG['paths']['output_root']) / 'evaluation' / 'summary.json'
    required_eval_df = pd.DataFrame(json.loads(summary_path.read_text())) if summary_path.exists() else pd.DataFrame()
required_eval_df

## 12. Ablation Plan and Evaluation

Ablations are trained by adding names to `TRAIN_EXPERIMENTS`. Evaluate completed ablations by adding names to `EVALUATE_ABLATIONS`.

In [ ]:
ablation_plan_df = pd.DataFrame([
    {'experiment': name, 'display_name': exp['display_name'], 'dataset_variant': exp['dataset_variant'], 'base_stage': exp['base_stage'], 'hypothesis': exp['hypothesis']}
    for name, exp in CONFIG['experiments'].items()
])
ablation_plan_df

In [ ]:
ablation_eval_frames = []
for experiment in EVALUATE_ABLATIONS:
    exp_cfg = CONFIG['experiments'][experiment]
    eval_cfg = {'base_stage': exp_cfg['base_stage'], 'adapter_experiment': experiment, 'shots': 0}
    result = evaluate_config(f'{experiment}_0shot', eval_cfg, CONFIG)
    ablation_eval_frames.append(save_eval_outputs(f'{experiment}_0shot', [result], CONFIG))

pd.concat(ablation_eval_frames, ignore_index=True) if ablation_eval_frames else pd.DataFrame()

## 13. Catastrophic Forgetting Probe

This compares OOD perplexity before and after SFT. Replace the placeholder OOD text with the same modern/Wikipedia slice used in HW1 before reporting final numbers.

In [ ]:
def load_ppl_model(config: Dict[str, Any], adapter_experiment: Optional[str]):
    model_path = str(project_path(config['paths']['part1_merged_model_dir']))
    tokenizer = load_tokenizer(model_path, config, padding_side='right')
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config(config),
        device_map=config['model']['device_map'],
        trust_remote_code=config['model'].get('trust_remote_code', False),
        attn_implementation=config['model'].get('attn_implementation', 'sdpa'),
    )
    if adapter_experiment:
        adapter_path = project_path(config['paths']['output_root']) / 'sft' / adapter_experiment / 'final_adapter'
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return model, tokenizer


def sliding_ppl(model, tokenizer, text: str, config: Dict[str, Any]) -> Dict[str, Any]:
    enc = tokenizer(text, return_tensors='pt')
    input_ids = enc.input_ids[:, :config['perplexity']['max_eval_tokens']].to(model.device)
    stride = config['perplexity']['stride']
    max_length = min(getattr(model.config, 'max_position_embeddings', 1024), 1024)
    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + max_length, input_ids.size(1))
        trg_len = end_loc - prev_end_loc
        input_chunk = input_ids[:, begin_loc:end_loc]
        target_ids = input_chunk.clone()
        target_ids[:, :-trg_len] = -100
        with torch.no_grad():
            nlls.append(model(input_chunk, labels=target_ids).loss)
        prev_end_loc = end_loc
        if end_loc == input_ids.size(1):
            break
    return {'perplexity': float(torch.exp(torch.stack(nlls).mean()).item()), 'windows': len(nlls), 'tokens': int(input_ids.size(1))}


def measure_ppl(label: str, adapter_experiment: Optional[str], config: Dict[str, Any]) -> Dict[str, Any]:
    text_path = project_path(config['perplexity']['ood_text_path'])
    if not text_path.exists():
        text_path.parent.mkdir(parents=True, exist_ok=True)
        text_path.write_text(('This is a modern encyclopedic prose placeholder for the OOD perplexity probe. Replace this file with the HW1 OOD slice.\n' * 300), encoding='utf-8')
    model, tokenizer = load_ppl_model(config, adapter_experiment)
    result = sliding_ppl(model, tokenizer, text_path.read_text(encoding='utf-8'), config)
    result['adapter_experiment'] = label
    out_dir = project_path(config['paths']['output_root']) / 'perplexity'
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / f'{label}.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return result

if RUN_OOD_PPL:
    ppl_df = pd.DataFrame([measure_ppl('part1_merged', None, CONFIG), measure_ppl('main', 'main', CONFIG)])
else:
    ppl_df = pd.DataFrame()
ppl_df

## 14. Report Assets

Collect key tables into a Markdown artifact for the report. Re-run this after evaluation and perplexity are complete.

In [ ]:
def make_report_assets(config: Dict[str, Any]) -> Path:
    assets_dir = project_path(config['paths']['report_assets_dir'])
    assets_dir.mkdir(parents=True, exist_ok=True)
    eval_summary_path = project_path(config['paths']['output_root']) / 'evaluation' / 'summary.json'
    eval_summary = json.loads(eval_summary_path.read_text()) if eval_summary_path.exists() else []
    lines = ['# HW2 Report Assets', '']
    lines += ['## Dataset Split', markdown_table(['split', 'count'], [['train', len(train_rows)], ['validation', len(validation_rows)], ['test', len(test_rows)]]), '']
    lines += ['## Required Evaluation']
    if eval_summary:
        lines.append(markdown_table(['configuration', 'accuracy', 'f1', 'n'], [[r['eval_config'], f"{r['accuracy']:.3f}", f"{r['f1']:.3f}", r['n']] for r in eval_summary]))
    else:
        lines.append('Evaluation summary not found yet.')
    lines.append('')
    ppl_dir = project_path(config['paths']['output_root']) / 'perplexity'
    ppl_rows = []
    if ppl_dir.exists():
        for path in sorted(ppl_dir.glob('*.json')):
            row = json.loads(path.read_text())
            ppl_rows.append([row['adapter_experiment'], f"{row['perplexity']:.3f}", row['tokens'], row['windows']])
    lines += ['## Catastrophic Forgetting Probe', markdown_table(['model', 'OOD perplexity', 'tokens', 'windows'], ppl_rows) if ppl_rows else 'Perplexity outputs not found yet.', '']
    lines += ['## Experiment Hypotheses', markdown_table(['experiment', 'display', 'hypothesis'], [[name, exp['display_name'], exp['hypothesis']] for name, exp in config['experiments'].items()])]
    out = assets_dir / 'report_assets.md'
    out.write_text('\n'.join(lines), encoding='utf-8')
    return out

report_assets_path = make_report_assets(CONFIG) if MAKE_REPORT_ASSETS else None
report_assets_path

## 15. Output Manifest

Expected outputs after the full cluster run.

In [ ]:
manifest_paths = [
    CONFIG['dataset']['dataset_card'],
    CONFIG['dataset']['train_messages'],
    CONFIG['dataset']['validation_messages'],
    CONFIG['dataset']['test_messages'],
    CONFIG['dataset']['test_answers'],
    CONFIG['paths']['part1_merged_model_dir'],
    'runs/sft/main/final_adapter',
    'runs/evaluation/summary.json',
    'runs/evaluation/predictions_required.jsonl',
    'report_assets/report_assets.md',
]
manifest = []
for item in manifest_paths:
    p = project_path(item)
    manifest.append({'path': str(p), 'exists': p.exists(), 'type': 'dir' if p.is_dir() else 'file' if p.is_file() else 'missing'})
pd.DataFrame(manifest)